# Logit Attribution Analysis

Fine-grained logit attribution: full decomposition, per-head attribution,
by-position attribution, promoted/suppressed tokens, and summary.

## Why This Matters

Logit attribution analyzes how the model's output predictions form and change across layers. Understanding logit formation — which components contribute what to the final prediction — is central to explaining model behavior.

**Key references:**
- [nostalgebraist (2020) "interpreting GPT: the logit lens"](https://www.lesswrong.com/posts/AcKRB8wDpdaN6v6ru/interpreting-gpt-the-logit-lens) — Project intermediate residuals through the unembedding
- [Belrose et al. (2023) "Eliciting Latent Predictions"](https://arxiv.org/abs/2303.08112) — Learned affine probes improve on the logit lens

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.logit_attribution_analysis import (
    full_logit_decomposition, per_head_logit_attribution,
    logit_attribution_by_position, top_promoted_suppressed,
    logit_attribution_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Full Logit Decomposition

Decompose target logit into embed, attention, MLP, and bias contributions.

In [ ]:
result = full_logit_decomposition(model, tokens)
print(f"Target token: {result['target_token']}, Total: {result['total_logit']:.4f}\n")
for c in result['components']:
    bar = '█' * int(abs(c['logit_contribution']) * 10)
    sign = '+' if c['logit_contribution'] >= 0 else '-'
    print(f"  {c['name']:12s}: {c['logit_contribution']:+.4f} {sign}{bar}")

## Per-Head Logit Attribution

Which attention heads contribute most to the prediction?

In [ ]:
result = per_head_logit_attribution(model, tokens)
print(f"Top positive: {', '.join(f'L{h[\'layer\']}H{h[\'head\']}' for h in result['top_positive'])}")
print(f"Top negative: {', '.join(f'L{h[\'layer\']}H{h[\'head\']}' for h in result['top_negative'])}\n")
for h in result['heads'][:8]:
    bar = '█' * int(abs(h['logit_contribution']) * 20)
    print(f"  L{h['layer']}H{h['head']}: {h['logit_contribution']:+.4f} (norm={h['output_norm']:.4f}) {bar}")

## Attribution by Position

Which source positions contribute most to the target logit?

In [ ]:
result = logit_attribution_by_position(model, tokens)
print(f"Target token: {result['target_token']}\n")
for s in result['per_source']:
    bar = '█' * int(abs(s['logit_contribution']) * 20)
    print(f"  Pos {s['position']} (tok {s['token_id']:3d}): {s['logit_contribution']:+.4f} {bar}")

## Top Promoted & Suppressed

Which vocabulary tokens get the highest and lowest logits?

In [ ]:
result = top_promoted_suppressed(model, tokens, top_k=5)
print(f"Logit range: {result['logit_range']:.4f}\n")
print('Promoted:')
for t in result['promoted']:
    print(f"  tok {t['token_id']:3d}: logit={t['logit']:+.4f}, prob={t['probability']:.4f}")
print('\nSuppressed:')
for t in result['suppressed']:
    print(f"  tok {t['token_id']:3d}: logit={t['logit']:+.4f}, prob={t['probability']:.6f}")

## Attribution Summary

Combined overview of logit attribution.

In [ ]:
result = logit_attribution_summary(model, tokens)
print(f"Target: tok {result['target_token']}, total logit: {result['total_logit']:.4f}")
print(f"Embed: {result['embed_contribution']:+.4f}")
print(f"Total attn: {result['total_attn_contribution']:+.4f}")
print(f"Total mlp: {result['total_mlp_contribution']:+.4f}")
print(f"Top head: {result['top_head']} ({result['top_head_contribution']:+.4f})")